#### This notebook converts RAG system into a production-style API.

 ### Install Required Packages

In [1]:
!pip install fastapi uvicorn

  Using cached uvicorn-0.49.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
Using cached uvicorn-0.49.0-py3-none-any.whl (71 kB)
Using cached starlette-1.3.1-py3-none-any.whl (73 kB)

   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fast

### Import Libraries

In [2]:
import pickle
import faiss
from fastapi import FastAPI
from pydantic import BaseModel

###  Load Metadata

In [3]:
with open("data/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print("Metadata Loaded:", len(metadata))

Metadata Loaded: 6


### Load Vectorizer

In [4]:
with open("data/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print("Vectorizer Loaded Successfully")

Vectorizer Loaded Successfully


###  Load FAISS Index

In [5]:
index = faiss.read_index("data/faiss_index.bin")

print("Vectors Stored:", index.ntotal)

Vectors Stored: 6


### Define Retrieval Function

In [6]:
def retrieve_context(question, k=3):

    query_vector = vectorizer.transform([question])
    query_vector = query_vector.toarray().astype("float32")

    distances, indices = index.search(query_vector, k)

    results = []

    for idx in indices[0]:

        results.append({
            "source": metadata[idx]["source"],
            "text": metadata[idx]["text"]
        })

    return results

### Define RAG Function

In [7]:
def generate_answer(question):

    results = retrieve_context(question)

    return {
        "question": question,
        "answer": results[0]["text"],
        "sources": [results[0]["source"]]
    }

### Create FastAPI App

In [8]:
app = FastAPI()

### Define Request Schema

In [9]:
class QuestionRequest(BaseModel):

    question: str

### Create API Endpoint

In [10]:
@app.post("/ask")

def ask_question(request: QuestionRequest):

    response = generate_answer(request.question)

    return response

### Save as app.py
Create a file named:

In [17]:
(app).py

AttributeError: 'FastAPI' object has no attribute 'py'